# TP final · Opción B · Comparación

Este notebook no entrena. Lee los runs terminados de `/Users/<usuario>/bigdata_emanuel_rb_tp_final`, los muestra como tabla y elige el de mayor F1 macro de test.

Entran los cuatro de `los_cuatro` y los que hayas agregado con `uno`. Si un nombre se repite, queda el run más reciente.

In [ ]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt

usuario = spark.sql("SELECT current_user()").first()[0]
experimento = f"/Users/{usuario}/bigdata_emanuel_rb_tp_final"
exp = mlflow.get_experiment_by_name(experimento)
if exp is None:
    raise RuntimeError(f"No existe el experimento {experimento}. Corré primero 01_experimentos.ipynb.")

runs = mlflow.search_runs([exp.experiment_id])
if runs.empty:
    raise RuntimeError(f"El experimento {experimento} no tiene runs. Corré primero 01_experimentos.ipynb.")
runs = runs[runs["status"] == "FINISHED"].copy()
if runs.empty:
    raise RuntimeError(f"El experimento {experimento} no tiene runs terminados.")
if "tags.mlflow.runName" in runs.columns:
    runs["run"] = runs["tags.mlflow.runName"]
else:
    runs["run"] = runs["run_name"]
runs = runs.sort_values("start_time", ascending=False).drop_duplicates("run")

columnas = {
    "run": "run",
    "params.modelo": "modelo",
    "params.que_entrenar": "que_entrenar",
    "params.balanceo": "balanceo",
    "params.C": "C",
    "params.n_arboles": "n_arboles",
    "params.profundidad": "profundidad",
    "params.puntajes_salud": "puntajes_salud",
    "metrics.train_f1_macro": "train_f1_macro",
    "metrics.test_f1_macro": "test_f1_macro",
    "metrics.test_accuracy": "test_accuracy",
    "metrics.test_f1_improve": "f1_improve",
    "metrics.test_f1_no_effect": "f1_no_effect",
    "metrics.test_f1_worsen": "f1_worsen",
}
for columna in columnas:
    if columna not in runs.columns:
        runs[columna] = None
comparacion = runs.rename(columns=columnas)[list(columnas.values())]
comparacion = comparacion.dropna(subset=["test_f1_macro"])
if comparacion.empty:
    raise RuntimeError("Ningún run tiene test_f1_macro. Volvé a 01_experimentos.ipynb.")
for metrica in ["train_f1_macro", "test_f1_macro", "test_accuracy", "f1_improve", "f1_no_effect", "f1_worsen"]:
    comparacion[metrica] = comparacion[metrica].astype(float)
comparacion = comparacion.sort_values("test_f1_macro", ascending=False).reset_index(drop=True)

print(f"Experimento: {experimento}")
print(f"Runs terminados: {len(comparacion)}")
display(comparacion.round(4))

nombres = ["logistica", "random_forest", "lightgbm", "catboost"]
faltan = [nombre for nombre in nombres if nombre not in set(comparacion["run"])]
if faltan:
    print(f"Todavía no están los runs de la comparación fija: {faltan}. Corré 01 con Qué entrenar en los_cuatro.")

## Cuál es el mejor

El criterio de la entrega es el F1 macro de test. La accuracy se muestra al lado: con `Improve` cerca del 74 % de las filas, acertar la clase mayoritaria sube esa métrica.

In [ ]:
ver = ["run", "modelo", "que_entrenar", "balanceo", "puntajes_salud", "C", "n_arboles", "profundidad", "train_f1_macro", "test_f1_macro", "test_accuracy"]
por_f1 = comparacion.sort_values("test_f1_macro", ascending=False).head(1)
por_accuracy = comparacion.sort_values("test_accuracy", ascending=False).head(1)
display(pd.concat([
    por_f1.assign(criterio="mejor F1 macro"),
    por_accuracy.assign(criterio="mejor accuracy"),
])[["criterio"] + ver].round(4))
elegido = por_f1.iloc[0]
print(
    f"Elegido: {elegido['run']} con F1 macro de test {float(elegido['test_f1_macro']):.4f}. "
    f"En train ese F1 es {float(elegido['train_f1_macro']):.4f}."
)
print(
    f"Por accuracy ganaría {por_accuracy.iloc[0]['run']} "
    f"({float(por_accuracy.iloc[0]['test_accuracy']):.4f}). La accuracy no decide."
)

## Los runs en gráficos

Cada color es un modelo. En el segundo gráfico, un punto lejos arriba de la diagonal tiene F1 de train mucho más alto que el de test: memorizó el corte.

In [ ]:
colores = {
    "logistica": "#2563eb",
    "random_forest": "#16a34a",
    "lightgbm": "#f59e0b",
    "catboost": "#7c3aed",
}
ordenado = comparacion.sort_values("test_f1_macro")
fig, axes = plt.subplots(1, 2, figsize=(12, max(3.5, 0.38 * len(comparacion) + 1.2)))
axes[0].barh(
    ordenado["run"],
    ordenado["test_f1_macro"],
    color=[colores.get(modelo, "#94a3b8") for modelo in ordenado["modelo"]],
)
axes[0].set_xlabel("F1 macro en test")
axes[0].set_title("Todos los runs")
axes[1].scatter(
    comparacion["train_f1_macro"],
    comparacion["test_f1_macro"],
    s=60,
    color=[colores.get(modelo, "#94a3b8") for modelo in comparacion["modelo"]],
)
for fila in comparacion.itertuples():
    axes[1].annotate(fila.run, (fila.train_f1_macro, fila.test_f1_macro), fontsize=7, xytext=(4, 3), textcoords="offset points")
ejes = [0, 1.02]
axes[1].plot(ejes, ejes, color="#94a3b8", ls="--")
axes[1].set_xlim(*ejes)
axes[1].set_ylim(*ejes)
axes[1].set_xlabel("F1 macro en train")
axes[1].set_ylabel("F1 macro en test")
axes[1].set_title("Lejos de la diagonal = sobreajuste")
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Tabla para la entrega

Copiá esta tabla si querés pegarla al lado de la corrida de Databricks. El número que vale es el de la tabla de arriba.

In [ ]:
copiar = ["run", "modelo", "que_entrenar", "balanceo", "puntajes_salud", "train_f1_macro", "test_f1_macro", "test_accuracy", "f1_improve", "f1_no_effect", "f1_worsen"]
vista = comparacion[copiar].round(4).fillna("")
lineas = ["| " + " | ".join(copiar) + " |", "|" + "|".join(["---"] * len(copiar)) + "|"]
for _, fila in vista.iterrows():
    lineas.append("| " + " | ".join(str(fila[col]) for col in copiar) + " |")
print("\n".join(lineas))

## Por qué ese modelo

El criterio estaba fijado antes de mirar la tabla: gana el mayor F1 macro de test. Las tres clases pesan igual. `Improve` es cerca del 74 % de las respuestas, así que un modelo que casi siempre elige esa clase sube la accuracy y puede quedar abajo en F1 macro.

En la corrida local (scikit-learn 1.9.1, LightGBM 4.7.0, CatBoost 1.2.10, sin puntajes de salud) el orden por F1 macro fue `random_forest` 0.3891, `lightgbm` 0.3775, `catboost` 0.3713, `logistica` 0.3387. Por accuracy iba primero `lightgbm` (0.7260) y el bosque quedaba en 0.7192. El F1 de `Worsen` fue 0 en los cuatro: el test de ese corte tiene 3 filas de esa clase y ninguno las acertó. El número que vale es el de la tabla de arriba, leída de MLflow.


## Limitaciones

La encuesta salió de un Google Form en Reddit, Discord y redes. Quien respondió eligió entrar. El modelo no dice qué le pasa a alguien que no contestó este formulario.

`music_effects` y los puntajes de 0 a 10 son lo que la persona escribió, no un diagnóstico.

`Worsen` tiene 17 filas en toda la encuesta y alrededor de 3 en el test. Con tan pocas, el F1 de esa clase se mueve si cambia la semilla, y el orden entre modelos también.

Los árboles de esta configuración memorizan el train: el F1 macro de train llega a 1 y el de test queda mucho más abajo. Los hiperparámetros están fijos. La elección es el mejor F1 macro de test en este corte, no una búsqueda ni un resultado sobre la población.
